In [0]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

!pip install holidays

In [0]:
df = pd.read_csv("../1_Dataset/Processed_data/1_dispatch_price.csv", index_col=0, parse_dates=True)
df_core_columns = df.columns
df_base = df
df_base[:10]

In [0]:
def _add_arcsinh_price_lags(df: pd.DataFrame) -> pd.DataFrame:
    """
    arcsinh-transformed price lags.
    arcsinh handles negatives and compresses extreme spikes,
    giving the model a better-scaled view of price history.
    """
    df = df.copy()
    scale = 100
    for col in df_core_columns:
        ap = np.arcsinh(df[col] / scale).rename("_ap")
        for lag in [1, 2, 4, 12, 48, 96, 336, 335, 337]:
            df[f"{col}_asinh_lag_{lag}"] = ap.shift(lag).astype(np.float32)
        df[f"{col}_asinh_rmean_48"]  = ap.rolling(48).mean().astype(np.float32)
        df[f"{col}_asinh_rmean_336"] = ap.rolling(336).mean().astype(np.float32)
    return df

new_df = _add_arcsinh_price_lags(df_base)

df = pd.concat([df, new_df.drop(columns=df_core_columns)], axis=1)

new_df[:10]


In [0]:
def _add_time_features(df: pd.DataFrame) -> pd.DataFrame:

    import holidays

    _NSW_HOLIDAYS = holidays.Australia(state="NSW", years=range(2018, 2026))
    
    idx = df.index
    df = df.copy()

    # Raw calendar components
    df["hour"]        = idx.hour.astype(np.int8)
    df["dayofweek"]   = idx.dayofweek.astype(np.int8)
    df["month"]       = idx.month.astype(np.int8)
    df["dayofyear"]   = idx.day_of_year.astype(np.int16)

    # Cyclical (sin/cos) encodings so the model sees periodicity
    df["hour_sin"]    = np.sin(2 * np.pi * idx.hour / 24).astype(np.float32)
    df["hour_cos"]    = np.cos(2 * np.pi * idx.hour / 24).astype(np.float32)
    df["dow_sin"]     = np.sin(2 * np.pi * idx.dayofweek / 7).astype(np.float32)
    df["dow_cos"]     = np.cos(2 * np.pi * idx.dayofweek / 7).astype(np.float32)
    df["month_sin"]   = np.sin(2 * np.pi * (idx.month - 1) / 12).astype(np.float32)
    df["month_cos"]   = np.cos(2 * np.pi * (idx.month - 1) / 12).astype(np.float32)

    # Binary flags
    df["is_weekend"]  = (idx.dayofweek >= 5).astype(np.float32)
    df["is_holiday"]  = np.array(
        [d.date() in _NSW_HOLIDAYS for d in idx], dtype=np.float32
    )
    # Peak (17–20 h) and off-peak (< 7 h or ≥ 21 h) periods
    df["is_peak"]     = ((idx.hour >= 17) & (idx.hour <= 20)).astype(np.float32)
    df["is_shoulder"] = ((idx.hour >= 7)  & (idx.hour < 17)).astype(np.float32)
    df["is_off_peak"] = ((idx.hour < 7)   | (idx.hour >= 21)).astype(np.float32)

    # Combined weekend/holiday flag — demand behaviour is quite different
    df["is_offday"]   = np.maximum(df["is_weekend"], df["is_holiday"]).astype(np.float32)

    return df

new_df = _add_time_features(df_base)

df = pd.concat([df, new_df.drop(columns=df_core_columns)], axis=1)

new_df[:10]


In [0]:
def _add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    LAG_INTERVALS = sorted(set([
        1, 2, 3, 4, 6, 8, 12, 24,
        48, 49, 50, 51, 52,
        96, 97, 98,
        # 3-day and 4-day same-period anchors — captures weekly envelope effects
        # without redundancy with the weekly (336) lag.
        144, 143, 145,
        192, 191, 193,
        336, 335, 337,
        672, 671, 673,
    ] + list(range(26, 72, 2))))  # even lags covering 13h–35h window

    new_cols = {}
    for col in df_core_columns:
        for lag in LAG_INTERVALS:
            new_cols[f"{col}_lag_{lag}"] = df[col].shift(lag).astype(np.float32)

    return pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)

new_df = _add_lag_features(df_base)

df = pd.concat([df, new_df.drop(columns=df_core_columns)], axis=1)

new_df[:10]


In [0]:
def _add_long_range_features(df: pd.DataFrame) -> pd.DataFrame:
    
    df   = df.copy()
    
    for col in df_core_columns:
       
        ap   = np.arcsinh(df[col] / 100)
        BASE = 17_532

        # --- Annual price lags (same period last year ± spread) ---
        for offset in range(-2, 2 + 1):
            lag = BASE + offset
            df[f"{col}_lag_annual_{'+' if offset >= 0 else ''}{offset}"] = (
                df[col].shift(lag).astype(np.float32)
            )
            df[f"{col}_asinh_lag_annual_{'+' if offset >= 0 else ''}{offset}"] = (
                ap.shift(lag).astype(np.float32)
            )

        p_annual = df[col].shift(BASE)
        df[f"{col}_annual_rmean_96"]  = p_annual.rolling(96, min_periods=24).mean().astype(np.float32)
        df[f"{col}_annual_rmax_96"]   = p_annual.rolling(96, min_periods=24).max().astype(np.float32)
        df[f"{col}_annual_rstd_96"]   = p_annual.rolling(96, min_periods=24).std().astype(np.float32)

        # Spike count over the same week last year
        df[f"{col}_annual_spike_96"]  = (p_annual >= 300).rolling(96, min_periods=24).sum().astype(np.float32)

        # Year-on-year price change (current vs same period last year)
        df[f"{col}_yoy_change"]  = (df[col] - df[col].shift(BASE)).astype(np.float32)
        df[f"{col}_yoy_ratio"]   = (df[col] / (df[col].shift(BASE).abs() + 1)).clip(0, 20).astype(np.float32)

        # Note: 2-week and 6-week rolling stats ({col}_rmean_672, _rmax_672, _rmin_672,
        # _rstd_672, _rmean_2016) are generated by _add_rolling_features to avoid duplication.

    return df

new_df = _add_long_range_features(df_base)

df = pd.concat([df, new_df.drop(columns=df_core_columns)], axis=1)

new_df[:10]


In [0]:
def _add_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    """Rolling statistics computed on price available at each timestamp."""
    ROLLING_WINDOWS = [4, 8, 24, 48, 96, 336, 672, 2016]

    new_cols = {}
    for col in df_core_columns:
        for w in ROLLING_WINDOWS:
            min_p = max(1, w // 2)
            rolled = df[col].rolling(w, min_periods=min_p)
            new_cols[f"{col}_rmean_{w}"] = rolled.mean().astype(np.float32)
            new_cols[f"{col}_rstd_{w}"]  = rolled.std().astype(np.float32)
            new_cols[f"{col}_rmax_{w}"]  = rolled.max().astype(np.float32)
            new_cols[f"{col}_rmin_{w}"]  = rolled.min().astype(np.float32)  

    return pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)

new_df = _add_rolling_features(df_base)

df = pd.concat([df, new_df.drop(columns=df_core_columns)], axis=1)

new_df[:10]


In [0]:
def _add_regime_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Capture price-spike and volatility regime signals.
    All windows look backward only, so there is no future leakage.
    Note: vol_48/vol_336 (rolling std) are already produced as rstd_48/rstd_336
    by _add_rolling_features and are not repeated here.
    """
    df = df.copy()

    for col in df_core_columns:
        
        # Was there a high-price spike in the last 24 h?
        df[f"{col}_spike_flag_48"]    = (df[col].rolling(48).max()  > 300).astype(np.float32)
        # Was there a negative price in the last 24 h?
        df[f"{col}_neg_flag_48"]      = (df[col].rolling(48).min()  < 0).astype(np.float32)
        # Quantile context (recent price level relative to weekly range)
        df[f"{col}_q90_336"]    = df[col].rolling(336).quantile(0.90).astype(np.float32)
        df[f"{col}_q10_336"]    = df[col].rolling(336).quantile(0.10).astype(np.float32)
        df[f"{col}_pct_rank_48"] = (
            df[col].rolling(48).rank(pct=True)
        ).astype(np.float32)

    return df

new_df = _add_regime_features(df_base)

df = pd.concat([df, new_df.drop(columns=df_core_columns)], axis=1)

new_df[:10]


In [0]:
def _add_time_since_spike_features(df: pd.DataFrame) -> pd.DataFrame:
   
    df   = df.copy()
    
    # 30-min intervals between rows
    INTERVAL_H = 0.5
    # Maximum hours to report (cap at 2 weeks to avoid unbounded values early in series)
    MAX_HOURS  = 336.0  # 2 weeks

    for col in df_core_columns:
        for threshold in [150, 300, 1000]:
            spike_flag = (df[col] >= threshold).astype(np.float32)

            positions       = pd.RangeIndex(len(df))
            last_spike_pos  = (
                pd.Series(np.where(spike_flag.values, positions, np.nan), index=df.index)
                .ffill()
                .fillna(-MAX_HOURS / INTERVAL_H)  # no prior spike seen → treat as max
            )
            intervals_since = (pd.Series(positions, index=df.index) - last_spike_pos).clip(upper=MAX_HOURS / INTERVAL_H)
            hours_since     = (intervals_since * INTERVAL_H).astype(np.float32)
            col_thr         = f"{col}_hours_since_spike_{threshold}"
            df[col_thr]     = hours_since

            # Log-scale version reduces the numeric range for the tree learner
            df[f"{col}_log1p_hours_since_spike_{threshold}"] = np.log1p(hours_since).astype(np.float32)

            # Binary flags: was there a spike at each lookback horizon?
            for lookback_h in [1, 6, 12, 24, 48, 168]:   # 1h, 6h, 12h, 24h, 48h, 1wk
                intervals = int(lookback_h / INTERVAL_H)
                df[f"{col}_spike_{threshold}_last_{lookback_h}h"] = (
                    spike_flag.rolling(intervals, min_periods=1).max().astype(np.float32)
                )

        # Combined: hours since any of the three thresholds (summarises overall stress state)
        df[f"{col}_hours_since_any_spike"] = df[[f"{col}_hours_since_spike_150",
                                        f"{col}_hours_since_spike_300",
                                        f"{col}_hours_since_spike_1000"]].min(axis=1).astype(np.float32)

    return df


new_df = _add_time_since_spike_features(df_base)

df = pd.concat([df, new_df.drop(columns=df_core_columns)], axis=1)

new_df[:10]


In [0]:
def _add_spike_predictors(df: pd.DataFrame) -> pd.DataFrame:
    """
    Price momentum and spike-history features.
    All look-back only — zero leakage.
    Note: pct_rank_48 is already produced by _add_regime_features and is not repeated here.
    """

    df = df.copy()

    for col in df_core_columns:

        # Price momentum — how fast prices are moving right now
        df[f"{col}_mom_4"]  = df[col].diff(4).astype(np.float32)    # 2h
        df[f"{col}_mom_12"] = df[col].diff(12).astype(np.float32)   # 6h
        df[f"{col}_mom_48"] = df[col].diff(48).astype(np.float32)   # 24h

        # Price acceleration (second derivative) — is the current spike accelerating?
        df[f"{col}_accel_4"]  = df[col].diff(4).diff(4).clip(-2000, 2000).astype(np.float32)
        df[f"{col}_accel_12"] = df[col].diff(12).diff(12).clip(-2000, 2000).astype(np.float32)

        # Spike and negative price counts in recent history
        df[f"{col}_spike_count_48"]  = (df[col] >= 300).rolling(48).sum().astype(np.float32)
        df[f"{col}_spike_count_336"] = (df[col] >= 300).rolling(336).sum().astype(np.float32)
        df[f"{col}_neg_count_48"]    = (df[col] < 0).rolling(48).sum().astype(np.float32)

        # Spike intensity: cumulative spike energy in last 24h (not just count)
        SPIKE_THR = 150.0
        spike_excess = (df[col] - SPIKE_THR).clip(lower=0)
        df[f"{col}_spike_intensity_48"]  = spike_excess.rolling(48).sum().astype(np.float32)
        df[f"{col}_spike_intensity_336"] = spike_excess.rolling(336).sum().astype(np.float32)

        # Price percentile rank over weekly window
        df[f"{col}_pct_rank_336"] = df[col].rolling(336).rank(pct=True).astype(np.float32)

    return df

new_df = _add_spike_predictors(df_base)

df = pd.concat([df, new_df.drop(columns=df_core_columns)], axis=1)

new_df[:10]


In [0]:
def _add_region_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pairwise spread features for all NEM region prices (NSW, QLD, VIC, SA).
    Captures interconnector pressure from any direction — useful for predicting any state.

    Lags, rolling stats, arcsinh lags, and spike counts are handled by their dedicated
    functions (_add_lag_features, _add_rolling_features, _add_arcsinh_price_lags,
    _add_spike_predictors) and are not repeated here.
    """

    df = df.copy()
    new_cols: dict = {}

    for col in df_core_columns:
        # Pairwise spreads vs every other region — captures interconnector pressure
        for other_col in df_core_columns:
            if other_col == col:
                continue
            spread = (df[col] - df[other_col]).astype(np.float32)
            new_cols[f"{col}_vs_{other_col}_spread"]      = spread
            new_cols[f"{col}_vs_{other_col}_spread_lag1"] = spread.shift(1).astype(np.float32)

    # Multi-region spike co-occurrence: all regions elevated simultaneously
    regions_present = [c for c in df_core_columns if c in df.columns]
    if len(regions_present) >= 2:
        flags = pd.concat([(df[c] >= 150).astype(np.float32) for c in regions_present], axis=1)
        new_cols["multi_region_spike"] = flags.min(axis=1)  # 1 only if ALL elevated
        new_cols["region_spike_count"] = flags.sum(axis=1).astype(np.float32)  # 0–4

    if new_cols:
        df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)
    return df

new_df = _add_region_features(df_base)

df = pd.concat([df, new_df.drop(columns=df_core_columns)], axis=1)

new_df[:10]


In [0]:
print("Total features:",df.shape[1])

df.to_parquet("Feature_data/1_dispatch_price.parquet")